Загрузка и профилирование

In [1]:
import os
import sqlite3
import pandas as pd

df_2009_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print(f"Всего строк: {len(df)}")
print(f"Столбцы: {list(df.columns)}")

Всего строк: 1067371
Столбцы: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB


Очистка столбцов (Invoice, StockCode, Description, Customer ID, Country, Revenue и даты)

In [3]:
def missing_report(df):
    
    """Считает пропуски в каждом столбце таблицы.

    Args:
        df: таблица pandas, которую нужно проверить.

    Returns:
        Таблица с двумя столбцами: missing_count (число пропусков) и
        missing_pct (их доля в процентах). Показаны только столбцы с
        пропусками, от большей доли к меньшей.
    """
    
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    report = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    return report[report["missing_count"] > 0].sort_values(
        "missing_pct", ascending=False
    )

missing_report(df)

,missing_count,missing_pct
Customer ID,243007,22.77
Description,4382,0.41


In [4]:
df["Invoice"] = df["Invoice"].astype(str)

df["IsCancelled"] = df["Invoice"].str.startswith("C")
df["IsBadDebt"] = df["Invoice"].str.startswith("A")

print("Отменённые (C): ", df["IsCancelled"].sum())
print("Bad Debt (A): ", df["IsBadDebt"].sum())

Отменённые (C):  19494
Bad Debt (A):  6


In [5]:
df["InvoiceNumber"] = df["Invoice"].str.replace("[A-Za-z]", "", regex=True)
df[df["IsCancelled"]][["Invoice", "InvoiceNumber", "IsCancelled"]].head(5)

,Invoice,InvoiceNumber,IsCancelled
178,C489449,489449,True
179,C489449,489449,True
180,C489449,489449,True
181,C489449,489449,True
182,C489449,489449,True


In [6]:
has_letters = df["InvoiceNumber"].str.contains("[A-Za-z]", regex=True)
assert has_letters.sum() == 0, "В InvoiceNumber остались буквы после очистки!"

print("Проверка пройдена: InvoiceNumber содержит только цифры")

Проверка пройдена: InvoiceNumber содержит только цифры


In [7]:
# Стандартный товарный код: 5 цифр и, возможно, одна буква на конце (85123, 85123A)
STANDARD_STOCKCODE_PATTERN = r"^\d{5}[A-Za-z]?$"
is_standard_stockcode = df["StockCode"].astype(str).str.match(STANDARD_STOCKCODE_PATTERN)

df["IsServiceCode"] = ~is_standard_stockcode
non_standard_stockrows = df[df["IsServiceCode"]]
print(f"Найдено нестандартных строк: {len(non_standard_stockrows)}")

Найдено нестандартных строк: 7466


In [8]:
non_standard_pattern = (
    non_standard_stockrows["StockCode"].astype(str).str.extract(r"([A-Za-z#]+)")
)
non_standard_pattern.value_counts()

0        
POST         2122
DOT          1446
M            1421
BL            923
C             283
LP            232
D             177
GR            123
DCGS          108
S             104
BANK          102
gift          100
bl             93
ADJUST         70
AMAZONFEE      43
DCGSSGIRL      25
DCGSSBOY       23
PADS           19
TEST           17
CRUK           16
B               6
m               5
SP              3
HC              1
GIFT            1
DCGSLBOY        1
DCGSLGIRL       1
J               1
Name: count, dtype: int64

In [9]:
known_service_codes = non_standard_stockrows["StockCode"].value_counts().index.tolist()
print(known_service_codes)

['POST', 'DOT', 'M', '15056BL', 'C2', '79323LP', 'D', '79323GR', 'S', 'BANK CHARGES', '15056bl', 'ADJUST', 'AMAZONFEE', 'DCGS0058', 'gift_0001_20', 'gift_0001_30', 'DCGSSGIRL', 'DCGSSBOY', 'PADS', 'gift_0001_10', 'CRUK', 'DCGS0076', 'TEST001', 'DCGS0003', 'gift_0001_50', 'gift_0001_40', 'DCGS0069', 'B', 'DCGS0004', 'm', 'gift_0001_80', 'DCGS0072', 'DCGS0066N', 'DCGS0068', 'gift_0001_70', 'DCGS0070', 'ADJUST2', 'SP1002', 'TEST002', 'gift_0001_60', 'gift_0001_90', 'DCGS0062', 'DCGS0037', 'DCGS0044', '72024HC', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0039', 'DCGS0060', 'DCGS0056', 'DCGS0059', 'GIFT', 'DCGSLBOY', 'DCGS0053', 'DCGSLGIRL', 'DCGS0075', 'DCGS0041', '47503J ', 'C3', 'DCGS0055', 'DCGS0074', 'DCGS0057', 'DCGS0073', 'DCGS0071', 'DCGS0066P', 'DCGS0067']


In [10]:
description_lookup = df[df["Description"].notna()].groupby("StockCode")["Description"].first()

print("StockCode хотя бы с одним описанием: ", len(description_lookup))
df["StockCode"] = df["StockCode"].astype(str)

StockCode хотя бы с одним описанием:  4950


In [11]:
df["Description"] = df["Description"].fillna(df["StockCode"].map(description_lookup))
print("Осталось пропусков после фильтрации: ", df["Description"].isna().sum())

Осталось пропусков после фильтрации:  3050


In [12]:
df["Description"] = df["Description"].fillna("Unknown")
df["HasUnknownDescription"] = df["Description"] == "Unknown"

unknown_rows = df[df["HasUnknownDescription"]]

print(
    "Распределение по Quantity: (положительные и отрицательные): ",
    (unknown_rows["Quantity"] < 0).value_counts(),
)
print()
print("IsCancelled среди строк: ", (unknown_rows["IsCancelled"]).value_counts())
print()
print(unknown_rows[["Invoice", "Quantity", "Price", "IsCancelled"]])

df["Description"] = df["Description"].astype(str)

Распределение по Quantity: (положительные и отрицательные):  Quantity
True     1873
False    1177
Name: count, dtype: int64

IsCancelled среди строк:  IsCancelled
False    3050
Name: count, dtype: int64

        Invoice  Quantity  Price  IsCancelled
470      489521       -50    0.0        False
3114     489655       -44    0.0        False
3161     489659       230    0.0        False
3731     489781        17    0.0        False
4296     489806      -770    0.0        False
...         ...       ...    ...          ...
1060783  581199        -2    0.0        False
1060787  581203        15    0.0        False
1060793  581209         6    0.0        False
1062442  581234        27    0.0        False
1064015  581408        20    0.0        False

[3050 rows x 4 columns]


In [13]:
df["IsStockAdjustment"] = df["HasUnknownDescription"] & (df["Price"] == 0)

print(df["IsStockAdjustment"].sum())

3050


In [14]:
df["HasUnknownDescription"] == df["IsStockAdjustment"].all()
# Флаги HasUnknownDescription и IsStockAdjustment совпадают по количеству строк

0          True
1          True
2          True
3          True
4          True
           ... 
1067366    True
1067367    True
1067368    True
1067369    True
1067370    True
Name: HasUnknownDescription, Length: 1067371, dtype: bool

In [15]:
print(df["Customer ID"].isna().sum())
print(len(df) - df["Customer ID"].count())
print(df["Customer ID"].value_counts())

num_unique_customers = df["Customer ID"].nunique()
print("Всего уникальных ID: ", num_unique_customers)

243007
243007
Customer ID
17841.0    13097
14911.0    11613
12748.0     7307
14606.0     6709
14096.0     5128
           ...  
14424.0        1
18184.0        1
13256.0        1
13017.0        1
15195.0        1
Name: count, Length: 5942, dtype: int64
Всего уникальных ID:  5942


In [16]:
df["IsGuest"] = df["Customer ID"].isna()

print("Всего гостевых заказов: ", df["IsGuest"].sum())

Всего гостевых заказов:  243007


In [17]:
countries_aliases = {"EIRE": "Ireland", "USA": "United States", "RSA": "South Africa"}
df["country_clean"] = df["Country"].replace(countries_aliases)
df["country_clean"].value_counts()

country_clean
United Kingdom          981330
Ireland                  17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
United States              535
Poland                     535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
South Africa             

In [18]:
non_standard_territories = [
    "Unspecified",
    "Channel Islands",
    "Hong Kong",
    "European Community",
    "West Indies",
    "Bermuda",
]
df["IsNotStandardGeography"] = df["country_clean"].isin(non_standard_territories)
print("Всего заказов из нестандартных территорий: ", df["IsNotStandardGeography"].sum())

Всего заказов из нестандартных территорий:  2933


In [19]:
df["Revenue"] = df["Price"] * df["Quantity"]
df["Revenue"].value_counts()

valid_sales_df = df[~df["IsServiceCode"] & ~df["IsCancelled"] & ~df["IsBadDebt"]]

print("Сумма всех строк: ", df["Revenue"].sum())
print("Сумма только валидных строк: ", valid_sales_df["Revenue"].sum())

Сумма всех строк:  19287250.568
Сумма только валидных строк:  20047603.31


In [20]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeekNum"] = df["InvoiceDate"].dt.dayofweek
df["DayOfWeekStr"] = df["InvoiceDate"].dt.day_name()

Проверки

In [21]:
flag_columns = [
    "IsCancelled",
    "IsBadDebt",
    "IsServiceCode",
    "IsStockAdjustment",
    "IsGuest",
    "IsNotStandardGeography",
]
for col in flag_columns:
    assert df[col].isna().sum() == 0, f"В флаге {col} есть пропуски!"

assert df["InvoiceNumber"].str.contains("[A-Za-z]", regex=True).sum() == 0, (
    "В InvoiceNumber есть строки с буквами!"
)

assert df["Revenue"].isna().sum() == 0, "В Revenue есть пропуски!"

assert df["Description"].isna().sum() == 0, "В Description есть пропуски!"

assert df["country_clean"].isna().sum() == 0, "В country_clean есть пропуски!"

assert df["country_clean"].isin(["EIRE", "RSA", "USA"]).sum() == 0, (
    "В country_clean остались старые значения!"
)

assert len(df) == len(df_2009_2010) + len(df_2010_2011), f("Ожидалось", len(df_2009_2010) + len(df_2010_2011)," , получили {len(df)}!")

assert df["Year"].isin([2009, 2010, 2011]).sum() == len(df), (
    "В Year есть лишние значения!"
)

print("Все проверки прошли успешно")

Все проверки прошли успешно


In [22]:
for col in df.columns:
    types_in_column = df[col].apply(type).nunique()
    if types_in_column > 1:
        print(f"{col}: обнаружено несколько типов: ", types_in_column)

Сохранение (Parquet, SQLite)

In [23]:
df.to_parquet("../data/processed/retail_cleaned.parquet", index=False)
print("Сохранено: ", df.shape)

Сохранено:  (1067371, 22)


In [24]:
conn = sqlite3.connect("../data/processed/retail.db")

df.to_sql("sales", conn, if_exists="replace", index=False)

print("Данные загружены в SQLite")

Данные загружены в SQLite


In [25]:
import re
from pathlib import Path

def load_queries(path="../sql/queries.sql"):
    """Читает .sql-файл и возвращает словарь {имя_запроса: текст}.

    Каждый запрос в файле начинается со строки '-- name: <имя>'.
    """
    text = Path(path).read_text(encoding="utf-8")
    blocks = re.split(r"--\s*name:\s*", text)[1:]
    queries = {}
    for block in blocks:
        name, sql = block.split("\n", 1)
        queries[name.strip()] = sql.strip()
    return queries

queries = load_queries()
print(list(queries))

['top_products', 'month_revenue', 'rank_countries', 'final_revenue', 'users_comparison', 'days_revenue']


SQL-анализ и экспорт

In [26]:
top_products_df = pd.read_sql(queries["top_products"], conn)
month_revenue_df = pd.read_sql(queries["month_revenue"], conn)
rank_countries_df = pd.read_sql(queries["rank_countries"], conn)
final_revenue_df = pd.read_sql(queries["final_revenue"], conn)
users_comparison_df = pd.read_sql(queries["users_comparison"], conn)
days_revenue_df = pd.read_sql(queries["days_revenue"], conn)

with pd.ExcelWriter("../reports/summary_tables.xlsx", engine="openpyxl") as writer:
    top_products_df.to_excel(writer, sheet_name="Top Products", index=False)
    month_revenue_df.to_excel(writer, sheet_name="Month Revenue", index=False)
    rank_countries_df.to_excel(writer, sheet_name="Rank Countries", index=False)
    final_revenue_df.to_excel(writer, sheet_name="Revenue", index=False)
    users_comparison_df.to_excel(writer, sheet_name="Users Comparison", index=False)
    days_revenue_df.to_excel(writer, sheet_name="Revenue by Days", index=False)

conn.close()